# ETL #

Extraction, Transformation and loading of Brazilian E-Commerce Data Set available from [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

### Change Working Directory ###

- Access current directory

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\mikee\\Desktop\\BrazilianECommerce\\Notebooks'

- Make the parent of the current directory the new current directory
    * os.path.dirname() gets the parent directory
    * os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


- Confirm the directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\mikee\\Desktop\\BrazilianECommerce'

## Extraction & Transformation

We have 8 different files to inspect for our database - in order to maintain the integrity of files I will extract and clean them separately.

In [11]:
#import libaries
import pandas as pd
import numpy as np


In [12]:
#extract csv files individually
df1 = pd.read_csv('Data/Raw/olist_customers_dataset.csv')
df2 = pd.read_csv('Data/Raw/olist_geolocation_dataset.csv')
df3 = pd.read_csv('Data/Raw/olist_order_items_dataset.csv')
df4 = pd.read_csv('Data/Raw/olist_order_payments_dataset.csv')
df5 = pd.read_csv('Data/Raw/olist_order_reviews_dataset.csv')
df6 = pd.read_csv('Data/Raw/olist_orders_dataset.csv')
df7 = pd.read_csv('Data/Raw/olist_products_dataset.csv')
df8 = pd.read_csv('Data/Raw/olist_sellers_dataset.csv')
df9 = pd.read_csv('Data/Raw/product_category_name_translation.csv')#This file is used to translate the product category names from Portuguese to English in products dataset.

In [13]:
#append all the dataframes into a list
dataframes = [df1, df2, df3, df4, df5, df6, df7, df8, df9]

In [14]:
#loop through and check for missing values in each dataframe
for i, df in enumerate(dataframes):
    missing_values = df.isnull().sum().sum()
    print(f"DataFrame {i+1} has {missing_values} missing values.")

DataFrame 1 has 0 missing values.
DataFrame 2 has 0 missing values.
DataFrame 3 has 0 missing values.
DataFrame 4 has 0 missing values.
DataFrame 5 has 145903 missing values.
DataFrame 6 has 4908 missing values.
DataFrame 7 has 2448 missing values.
DataFrame 8 has 0 missing values.
DataFrame 9 has 0 missing values.


Dataframe 5 has by far the most missing values, but has these represent reviews that is to be expected. The orders and products datasets also have a number of missing values, before proceeding I will examine what these represent.

In [15]:
#check what columns have missing values in df6
df6.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [16]:
#check what columns have missing values in df7
df7.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

As the missing values do not represent critical values there is no need to drop any rows or columns.

The only dataset which needs transforming is df7, products, where we can use the product_category_name_translation.csv to translate portuguese product categories into english versions.

In [17]:
## translate the product category names from Portuguese to English in products dataset using df9
df7 = df7.merge(df9, left_on='product_category_name', right_on='product_category_name', how='left')

In [18]:
df7.head(3)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure


In [19]:
#drop portuguese product category name column and rename the english product category name column
df7.drop('product_category_name', axis=1, inplace=True)
df7.rename(columns={'product_category_name_english': 'product_category_name'}, inplace=True)

In [20]:
df7.head(3)

,product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name
0,1e9e8ef04dbcff4541ed26657ea517e5,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure


In [21]:
df7.columns

Index(['product_id', 'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'product_category_name'],
      dtype='str')

## LOADING ##

Load cleaned datasets to new csvs (Only really necessary for products dataset, but an opportunity to rename as more sql friendly table names)

In [22]:
#load the cleaned dataframes into a new folder called "Data/Cleaned" - rename to simple table names
df1.to_csv('Data/Cleaned/customers.csv', index=False)
df2.to_csv('Data/Cleaned/geolocation.csv', index=False)
df3.to_csv('Data/Cleaned/order_items.csv', index=False)
df4.to_csv('Data/Cleaned/order_payments.csv', index=False)
df5.to_csv('Data/Cleaned/order_reviews.csv', index=False)
df6.to_csv('Data/Cleaned/orders.csv', index=False)
df7.to_csv('Data/Cleaned/products.csv', index=False)    
df8.to_csv('Data/Cleaned/sellers.csv', index=False)